# Size overhead for OmniSphinx

In [1]:
from IPython.display import display, HTML

import math
from typing import Callable, NamedTuple
from functools import partial

In [2]:
# All values in this notebook are in bytes!

KAPPA = 16
# Point size for sec224p1, as chosen in the paper
EC_POINT_SIZE = 29
MAC_SIZE = KAPPA

# Inspired by Sphinx, we assume that we address a node within KAPPA bytes
NODE_ADDRESS_LENGTH = KAPPA
# Inspired by Sphinx, we assume that we address a recipient with 2*KAPPA bytes
DESTINATION_ADDRESS_LENGTH = 2 * KAPPA

## Sphinx

In [3]:
def hs_native_sphinx(path_length: int) -> int:
    pre_exit = (path_length - 1) * (NODE_ADDRESS_LENGTH + MAC_SIZE)
    exit = DESTINATION_ADDRESS_LENGTH
    return EC_POINT_SIZE + MAC_SIZE + pre_exit + exit

def hs_os_sphinx(path_length: int) -> int:
    relay_bytes = (
        4 # ConcatBytes
        + 3 # Hash
        + 4 # Decrypt
        + 4 + NODE_ADDRESS_LENGTH # Load
        + 2 # Forward
        + 1 # Stop
        + MAC_SIZE
    )
    exit_bytes = (
        4 # ConcatBytes
        + 3 # Hash
        + 4 # Decrypt
        + 4 # CutBytes
        + 3 # CreateZeroes
        + 3 # Verify
        + 4 # CutBytes
        + 2 # Forward
        + 1 # Stop
    )
    return EC_POINT_SIZE + MAC_SIZE + (path_length - 1) * relay_bytes + exit_bytes

def onion_size_native_sphinx(path_length: int, payload_size: int) -> int:
    # Sphinx adds kappa '0' to the payload to detect modifications
    return hs_native_sphinx(path_length) + KAPPA + payload_size

def onion_size_emulated_sphinx(path_length: int, payload_size: int) -> int:
    return hs_os_sphinx(path_length) + KAPPA + payload_size

## AE-Sphinx

In [4]:
def hs_native_aesphinx(path_length: int) -> int:
    # The trick in AE Sphinx is that the MAC in the header is also used to verify the integrity of the payload.
    # Therefore, no extra overhead is added compared to native Sphinx.
    return hs_native_sphinx(path_length)

def hs_os_aesphinx(path_length: int) -> int:
    relay_bytes = (
        4 + MAC_SIZE # Load
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # MAC
        + 3 # Verify
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # Decrypt
        + 4 + NODE_ADDRESS_LENGTH # Load
        + 2 # Forward
        + 1 # Stop
        + MAC_SIZE
    )
    exit_bytes = (
        4 + MAC_SIZE # Load
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # MAC
        + 3 # Verify
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # Decrypt
        + 4 + DESTINATION_ADDRESS_LENGTH # Load
        + 2 # Forward
        + 1 # Stop
    )
    return EC_POINT_SIZE + MAC_SIZE + (path_length - 1) * relay_bytes + exit_bytes

def onion_size_native_aesphinx(path_length: int, payload_size: int) -> int:
    return hs_native_aesphinx(path_length) + payload_size

def onion_size_emulated_aesphinx(path_length: int, payload_size: int) -> int:
    return hs_os_aesphinx(path_length) + payload_size

## EROR

In [5]:
def hs_native_eror(path_length: int) -> int:
    # This is the size of a single c^i
    per_hop_payload = EC_POINT_SIZE + 4      + KAPPA + max(DESTINATION_ADDRESS_LENGTH, NODE_ADDRESS_LENGTH)
    #                 ^- PKE overhead ^- role  ^- key  ^- address of hop/recipient
    
    # This is the size of a B_N^i
    per_hop_size = MAC_SIZE + per_hop_payload

    return path_length * per_hop_size

def hs_os_eror(path_length: int) -> int:
    per_hop_instructions = (
        5 # CutBytes
        + 5 # CutBytes
        + 5 # CutBytes
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 + MAC_SIZE # Load
        + 5 # MAC
        + 3 # Verify
        + 4 # Decrypt
        + 4 # Decrypt
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # PRG
        + 4 # XOR
        + 4 # Concat
        + 4 # Concat
        + 4 + max(DESTINATION_ADDRESS_LENGTH, NODE_ADDRESS_LENGTH) # Load
        + 2 # Forward
        + 1 # Stop
        + MAC_SIZE
    )
    return EC_POINT_SIZE + MAC_SIZE + path_length * per_hop_instructions

def onion_size_native_eror(path_length: int, payload_size: int) -> int:
    # EROR doubles the payload, and has an extra MAC for the backwards payload
    return hs_native_eror(path_length) + 2 * payload_size + MAC_SIZE

def onion_size_emulated_eror(path_length: int, payload_size: int) -> int:
    return hs_os_eror(path_length) + 2 * payload_size + MAC_SIZE

## MultiSphinx

In [6]:
def hs_native_multisphinx(path_length: int) -> int:
    # "For anyone other than the designated multiplication node,
    # MultiSphinx messages are indistinguishable from regular Sphinx packets."
    # -> They have the same header size, all the magic happens in the payload
    return hs_native_sphinx(path_length)

def hs_os_multisphinx(path_length: int) -> int:
    # These are the same as for AE-Sphinx, as MultiSphinx is based on that
    relay_bytes = (
        4 + MAC_SIZE # Load
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # MAC
        + 3 # Verify
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # Decrypt
        + 4 + NODE_ADDRESS_LENGTH # Load
        + 2 # Forward
        + 1 # Stop
        + MAC_SIZE
    )
    exit_bytes = (
        4 + MAC_SIZE # Load
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # MAC
        + 3 # Verify
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # Decrypt
        + 4 + DESTINATION_ADDRESS_LENGTH # Load
        + 2 # Forward
        + 1 # Stop
    )
    # This is new for MultiSphinx:
    multiplication_bytes = (
        4 + MAC_SIZE # Load
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # MAC
        + 3 # Verify
        + 4 # ConcatBytes
        + 3 # Hash
        + 4 # PRG
        + 4 # XOR
        + 4 + 2 # Load
        + 4 + 2 # Load
        + 4 + 2 # Load
        + 4 + 2 # Load
        + 4 + 2 # Load
        + 3 # Copy
        + 3 # For
        + 4 # CutBytes*
        + 4 # CutBytes*
        + 4 # CutBytes*
        + 4 # CutBytes*
        + 4 # CutBytes*
        + 2 # Forward
        + 1 # Stop
    )

    # We either have a multiplication node, or an exit node in a path
    return EC_POINT_SIZE + MAC_SIZE + (path_length - 1) * relay_bytes + max(exit_bytes, multiplication_bytes)

def onion_size_native_multisphinx(path_length: int, payload_size: int, replication: int) -> int:
    # MultiSphinx wraps complete inner packets
    inner_size = hs_native_multisphinx(path_length) + payload_size
    return hs_native_multisphinx(path_length) + replication * inner_size

def onion_size_emulated_multisphinx(path_length: int, payload_size: int, replication: int) -> int:
    inner_size = hs_os_multisphinx(path_length) + payload_size
    return hs_os_multisphinx(path_length) + replication * inner_size

## PolySphinx

In [7]:
def hs_native_polysphinx_post(path_length: int, replication: int) -> int:
    size_per_hop = 1       + KAPPA + NODE_ADDRESS_LENGTH + MAC_SIZE
    #              ^- flag   ^- key  ^- next hop           ^- next mac
    key_tree_path = math.ceil(math.log(replication, 2)) * path_length
    # key_tree_path is in bits right now, let's round up to bytes
    key_tree_path = int(8 * math.ceil(key_tree_path / 8))
    final_hop_size = 1      + DESTINATION_ADDRESS_LENGTH + KAPPA           + key_tree_path
    #                ^- flag                               ^- key tree seed
    return EC_POINT_SIZE + MAC_SIZE + (path_length - 1) * size_per_hop + final_hop_size

def hs_native_polysphinx_pre(path_length: int, replication: int) -> int:
    size_per_hop = 1 + KAPPA + NODE_ADDRESS_LENGTH + MAC_SIZE
    per_replication = NODE_ADDRESS_LENGTH + KAPPA + hs_native_polysphinx_post(path_length, replication)

    return EC_POINT_SIZE + MAC_SIZE + (path_length - 1) * size_per_hop + 1 + replication * per_replication

def hs_os_polysphinx_post(path_length: int, replication: int) -> int:
    relay_bytes = (
        4 + KAPPA # Load
        + 4 # Encrypt
        + 4 + NODE_ADDRESS_LENGTH # Load
        + 2 # Forward
        + 1 # Stop
        + MAC_SIZE
    )
    exit_bytes = (
        4 + KAPPA # Load
        + 4 + path_length # Load
        + 4 + DESTINATION_ADDRESS_LENGTH # Load
        + 3 # Hash
        + 3 # Hash
        + 3 # For
        + 5 # CutBytes
        + 4 # Add
        + 3 # Hash
        + 4 # Concat
        + 3 # For
        + 5 # CutBytes
        + 4 # Decrypt
        + 2 # Forward
        + 1 # Stop
    )

    return EC_POINT_SIZE + MAC_SIZE + (path_length - 1) * relay_bytes + exit_bytes

def hs_os_polysphinx_pre(path_length: int, replication: int) -> int:
    relay_bytes = (
        4 + KAPPA # Load
        + 4 # Encrypt
        + 4 + NODE_ADDRESS_LENGTH # Load
        + 2 # Forward
        + 1 # Stop
        + MAC_SIZE
    )
    replication_bytes = (
        3 # Copy
        + 4 + replication * hs_os_polysphinx_post(path_length, replication) # Load
        + 3 # For
        + 5 # CutBytes
        + 5 # CutBytes
        + 5 # CutBytes
        + 5 # CutBytes
        + 5 # CutBytes
        + 4 # Encrypt
        + 2 # Forward
        + 1 # Stop
    )

    return EC_POINT_SIZE + MAC_SIZE + (path_length - 1) * relay_bytes + replication_bytes

def onion_size_native_polysphinx(path_length: int, payload_size: int, replication: int) -> int:
    return hs_native_polysphinx_pre(path_length, replication) + payload_size

def onion_size_emulated_polysphinx(path_length: int, payload_size: int, replication: int) -> int:
    return hs_os_polysphinx_pre(path_length, replication) + payload_size

## Results

In [8]:
class Format(NamedTuple):
    name: str
    header_size_native: Callable
    header_size_emulated: Callable
    onion_size_native: Callable
    onion_size_emulated: Callable
    
PATH_LENGTH = 5
FORMATS = [
    Format("Sphinx", hs_native_sphinx, hs_os_sphinx, onion_size_native_sphinx, onion_size_emulated_sphinx),
    Format("AE-Sphinx", hs_native_aesphinx, hs_os_aesphinx, onion_size_native_aesphinx, onion_size_emulated_aesphinx),
    Format("EROR", hs_native_eror, hs_os_eror, onion_size_native_eror, onion_size_emulated_eror),
    Format(
        "MultiSphinx (p = 3)",
        hs_native_multisphinx,
        hs_os_multisphinx,
        partial(onion_size_native_multisphinx, replication=3),
        partial(onion_size_emulated_multisphinx, replication=3),
    ),
    Format(
        "PolySphinx (p = 3)",
        partial(hs_native_polysphinx_pre, replication=3),
        partial(hs_os_polysphinx_pre, replication=3),
        partial(onion_size_native_polysphinx, replication=3),
        partial(onion_size_emulated_polysphinx, replication=3),
    ),
]

def html_header_sizes(formats, path_length):
    rows = []
    for form in formats:
        cost_native = form.header_size_native(path_length)
        cost_emulated = form.header_size_emulated(path_length)
        diff = cost_emulated - cost_native
        diff_perc = round((cost_emulated / cost_native - 1) * 100, 0)
        row = f"""<tr>
            <td>{form.name}</td>
            <td>{cost_native} B</td>
            <td>{cost_emulated} B</td>
            <td>+{diff} B</td>
            <td>+{diff_perc:.0f}%</td>
        </tr>"""
        rows.append(row)

    header = "<tr><th></th><th>Native</th><th>Emulated</th><th>&Delta;</th><th>&Delta;%</th></tr>"
    html = "<table>" + header + "".join(rows) + "</table>"
    return html

header_table = html_header_sizes(FORMATS, PATH_LENGTH)

def html_onion_sizes(formats, path_length, payload_size):
    rows = []
    for form in formats:
        cost_native = form.onion_size_native(path_length, payload_size)
        cost_emulated = form.onion_size_emulated(path_length, payload_size)
        diff = cost_emulated - cost_native
        row = f"<tr><td>{form.name}</td><td>{cost_native} B</td><td>{cost_emulated} B</td><td>+{diff} B</td></tr>"
        rows.append(row)

    header = "<tr><th></th><th>Native</th><th>Emulated</th><th>&Delta;</th></tr>"
    html = "<table>" + header + "".join(rows) + "</table>"
    return html

os_table = html_onion_sizes(FORMATS, PATH_LENGTH, 1024)

display(HTML(f"""
<strong>Headers only</strong>
{header_table}
<br>
<strong>Onion size (1 KiB payload)</strong>
{os_table}"""))